In [ ]:
import random
import pygame
from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.1.246")
 
pygame.init()
WIDTH, HEIGHT = 600, 400
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("UGOT Driving Game")
font = pygame.font.SysFont(None, 32)
 
ROAD_WIDTH = 400 
ROAD_LEFT = (WIDTH - ROAD_WIDTH) // 2
ROAD_RIGHT = ROAD_LEFT + ROAD_WIDTH
LANE_MARK_WIDTH = 8
LANE_MARK_HEIGHT = 40
LANE_MARK_SPACING = 30

CAR_WIDTH = 40
CAR_HEIGHT = 60
CAR_COLOR = (200, 30, 30)

OBSTACLE_WIDTH = 50
OBSTACLE_HEIGHT = 40
OBSTACLE_COLOR = (30, 30, 180)
OBSTACLE_SPEED = 5
SPAWN_INTERVAL = 900
 
MAX_X = ROAD_RIGHT - 20
MIN_X = ROAD_LEFT + 20
MAX_Y = HEIGHT - 20
MIN_Y = 20
 
def draw_road(scroll_offset):
    screen.fill((80, 170, 80)) # background
    pygame.draw.rect(screen, (40, 40, 40), (ROAD_LEFT, 0, ROAD_WIDTH, HEIGHT)) # road
    pygame.draw.rect(screen, (255, 255, 255), (ROAD_LEFT, 0, 10, HEIGHT)) # left line
    pygame.draw.rect(screen, (255, 255, 255), (ROAD_RIGHT-10, 0, 10, HEIGHT)) # right line
 
    for y in range(-LANE_MARK_HEIGHT, HEIGHT, LANE_MARK_HEIGHT+LANE_MARK_SPACING):
        draw_y = y + scroll_offset % (LANE_MARK_HEIGHT + LANE_MARK_SPACING)
        pygame.draw.rect(screen, (255, 255, 0), (WIDTH//2-LANE_MARK_WIDTH//2, draw_y,
                        LANE_MARK_WIDTH, LANE_MARK_HEIGHT))
 
def draw_car(x, y):
    rect = pygame.Rect(int(x), int(y), CAR_WIDTH, CAR_HEIGHT)
    pygame.draw.rect(screen, CAR_COLOR, rect)

def draw_obstacle(obstacle):
    pygame.draw.rect(screen, (30, 30, 180), obstacle)
    pygame.draw.rect(screen, (255, 255, 255), obstacle, 2) # outline
 
def show_text(message, y, color=(255, 255, 255), font_obj = None):
    if font_obj is None:
        font_obj = font
    text_surface = font_obj.render(message, True, color)
    screen.blit(text_surface, (WIDTH // 2 - text_surface.get_width()//2, y))

x, y = WIDTH // 2, HEIGHT // 2 
speed = 7
 
def clamp(value, low, high):
    """Clamp a value between the low and high."""
    return max(low, min(high, value))
 
def read_gyro():
    data = got.read_gyro_data()
    pitch = data[0]
    roll = data[1]
    yaw = data[2]
    return pitch, roll, yaw
 
center_pitch, center_roll, center_yaw = read_gyro()
scroll_offset = 0
score = 0
game_over = False
last_spawn = pygame.time.get_ticks()
obstacles = []
 
running = True
while running:
    scroll_offset += speed
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE:
                center_pitch, center_roll, center_yaw = read_gyro()
                x, y = WIDTH // 2, HEIGHT // 2 
                if game_over:
                    game_over = False
                    x = WIDTH // 2 - CAR_WIDTH // 2
                    score = 0
                    obstacles.clear()
                    last_spawn = pygame.time.get_ticks()
    
    if not game_over:
        pitch, roll, yaw = read_gyro()
    
        move_x = (roll - center_roll) / 20 # left / right
        move_y = (pitch - center_pitch) / -20 # up / down
    
        x += move_x * speed
        y += move_y * speed
    
        x = clamp(x, MIN_X, MAX_X)
        y = clamp(y, MIN_Y, MAX_Y)

        score += 1
        now = pygame.time.get_ticks()
        if now - last_spawn > SPAWN_INTERVAL: # spawn new obstacle
            last_spawn = now
            lane_x = random.choice([ROAD_LEFT + 40, WIDTH//2 - OBSTACLE_WIDTH//2, 
                                    ROAD_RIGHT - 40 - OBSTACLE_WIDTH])
            obstacles.append(pygame.Rect(lane_x, -OBSTACLE_HEIGHT, OBSTACLE_WIDTH, OBSTACLE_HEIGHT))
        
        for obstacle in obstacles:
            obstacle.y += OBSTACLE_SPEED # update position
        
        # remove obstacles if off screen
        obstacles = [obs for obs in obstacles if obs.y < HEIGHT + OBSTACLE_HEIGHT]

        player_rect = pygame.Rect(int(x), int(y), CAR_WIDTH, CAR_HEIGHT)
        if any(player_rect.colliderect(obs) for obs in obstacles):
            game_over = True
 
    # screen.fill((255, 255, 255))
    draw_road(scroll_offset)
    # pygame.draw.circle(screen, (0, 255, 0), (int(x), int(y)), 20)
    draw_car(x, y)
    for obstacle in obstacles:
        draw_obstacle(obstacle)
 
    show_text(f"Score: {score}", 10)
    show_text("Press SPACE to center gyro", 30, (240, 240, 240))
    if game_over:
        show_text("GAME OVER", HEIGHT // 2, (255, 80, 80))
        show_text("Press SPACE to restart", HEIGHT // 2 + 20, (255, 80, 80))
    pygame.display.flip()
 
pygame.quit()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.4)
Hello from the pygame community. https://www.pygame.org/contribute.html
192.168.1.209:50051
